# 嬛嬛Chat PEFT实战

## 配置下环境

In [1]:
# % conda create -n peft python=3.12
# % conda activate peft
# # 升级pip
# %python -m pip install --upgrade pip
# # 更换 pypi 源加速库的安装
# %pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple

# pip install modelscope==1.16.1
# pip install transformers==4.43.1
# pip install accelerate==0.32.1
# pip install peft==0.11.1
# pip install datasets==2.20.0

In [2]:
data_path = "./data/嬛嬛chat/huanhuan.json"

In [3]:
from datasets import Dataset
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer, GenerationConfig
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

e:\anaconda3\envs\peft\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from modelscope import snapshot_download

# 指定模型 ID 和下载到本地的路径
model_dir = snapshot_download(
    'Qwen/Qwen2.5-0.5B-Instruct',
    local_dir='./models/Qwen2.5-0.5B-Instruct', # 精确映射到你代码里的路径
    ignore_patterns=[
        "*.pth"
    ]
)
print(f"模型已成功下载至: {model_dir}")



2026-05-19 14:00:03,792 - modelscope - INFO - Target directory already exists, skipping creation.


模型已成功下载至: ./models/Qwen2.5-0.5B-Instruct


In [5]:
# 这个版本是自己写special token
def process_func_version1(example):
    MAX_LENGTH = 384    # Llama分词器会将一个中文字切分为多个token，因此需要放开一些最大长度，保证数据的完整性
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer(f"""
                            <|begin_of_text|>
                            
                            <|start_header_id|>system<|end_header_id|>\n\n
                            Cutting Knowledge Date: December 2023\n
                            Today Date: 26 Jul 2024\n\n
                            现在你要扮演皇帝身边的女人--甄嬛
                            <|eot_id|>
                            
                            <|start_header_id|>user<|end_header_id|>\n\n
                            {example['instruction'] + example['input']}
                            <|eot_id|>
                            
                            <|start_header_id|>assistant<|end_header_id|>\n\n""",  # 这里加入了assistant 开头
                            add_special_tokens=False)  # add_special_tokens 不在开头加 special_tokens
    
    # 这里
    response = tokenizer(f"{example['output']}<|eot_id|>", add_special_tokens=False)

    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.eos_token_id]
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]  # 因为eos token咱们也是要关注的所以 补充为1
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.eos_token_id]
    
    # 截断长度
    if len(input_ids) > MAX_LENGTH:  # 做一个截断
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
        
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [6]:
def process_func(example):
    MAX_LENGTH = 384
    
    system_prompt = "现在你要扮演皇帝身边的女人--甄嬛"
    user_prompt = example["instruction"] + example["input"]
    assistant_answer = example["output"]
    
    # 1. 只构造 prompt 部分：system + user + assistant 开头
    prompt_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True # 这里加入了Assistant 开头，即<|start_header_id|>assistant<|end_header_id|>
    )
    
    # 2. 构造完整对话：system + user + assistant answer
    full_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_answer},
    ]
    
    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False
    )
    
    # 3. 分别 tokenize，用 prompt 长度来 mask labels
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False
    )["input_ids"]
    
    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH
    )
    
    input_ids = full["input_ids"]
    attention_mask = full["attention_mask"]
    labels = [-100] * len(prompt_ids) + input_ids[len(prompt_ids):]
    
    # 4. 如果被截断了，labels 也要同步截断
    labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }
    
    
    

In [7]:
print(torch.cuda.is_available())

True


In [8]:
device = "cuda:0"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# device map = Auto 有个好处模型太大， 他可以分块存储到显存，内存甚至磁盘
model = AutoModelForCausalLM.from_pretrained("./models/Qwen2.5-0.5B-Instruct",
                                             torch_dtype=torch.bfloat16,
                                             quantization_config=bnb_config,
                                             device_map="auto",)

# 开启梯度检查点时，要执行该方法
model.enable_input_require_grads() 

print(model.hf_device_map)

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained('./models/Qwen2.5-0.5B-Instruct', use_fast=False, trust_remote_code=True)

tokenizer.pad_token = tokenizer.eos_token

# 读取数据
df = pd.read_json(data_path)
ds = Dataset.from_pandas(df)
ds[3]

Exception in thread Thread-11 (_readerthread):
Traceback (most recent call last):
  File "e:\anaconda3\envs\peft\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "e:\anaconda3\envs\peft\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "e:\anaconda3\envs\peft\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb2 in position 7: invalid start byte


{'': 0}


{'instruction': '嬛妹妹，我虽是一介御医，俸禄微薄，可是我保证会一生一世对你好，疼爱你，保护你，永远事事以你为重。本来没半月一次到府上去请脉，能够偶尔见一次妹妹的笑靥，已经心满意足了，可谁知——而且我也知道，妹妹心里是不愿意去殿选的。',
 'input': '',
 'output': '实初哥哥这么说，就枉顾我们一直以来的兄妹情谊了，嬛儿没有哥哥，一直把你当作自己的亲哥哥一样看待，自然相信哥哥会待妹妹好的——自然了，以后有了嫂子，你也会对嫂子更好。'}

In [11]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
)

model = get_peft_model(model, lora_config)
print("===== LoRA modules =====")
for name, module in model.named_modules():
    if "lora" in name.lower():
        print(name, module)
args = TrainingArguments(
        output_dir="./output/Qwen2.5-0.5B-Instruct_lora",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4, # 累计4个半期，每更新一次梯度
        logging_steps=10, # 更新10次参数打印一次日志
        num_train_epochs=3, # 训练三个epoch
        save_steps=100, # 更新100次参数以后，保存一次 checkpoint
        save_on_each_node=True, # 每个机器都保存一次 checkpoint
        gradient_checkpointing=True # 保存一些中间结果，然后需要中间的一些梯度的时候，就从最近的保存点里面重新前向计算
    )


trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_ds,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    )
trainer.train() # 开始训练 






































































Map: 100%|██████████| 3729/3729 [00:08<00:00, 425.75 examples/s]


===== LoRA modules =====
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_dropout ModuleDict(
  (default): Dropout(p=0.1, inplace=False)
)
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_dropout.default Dropout(p=0.1, inplace=False)
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A ModuleDict(
  (default): Linear(in_features=896, out_features=8, bias=False)
)
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default Linear(in_features=896, out_features=8, bias=False)
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B ModuleDict(
  (default): Linear(in_features=8, out_features=896, bias=False)
)
base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default Linear(in_features=8, out_features=896, bias=False)
base_model.model.base_model.model.base_model.model

 95%|█████████▍| 2645/2796 [2:31:46<08:39,  3.44s/it]
                                                   
  0%|          | 10/2796 [00:24<1:51:02,  2.39s/it] 

{'loss': 4.0642, 'grad_norm': 15.18689250946045, 'learning_rate': 4.982117310443491e-05, 'epoch': 0.01}


                                                   
  1%|          | 20/2796 [00:48<1:50:06,  2.38s/it] 

...

{'loss': 3.4021, 'grad_norm': 12.51982593536377, 'learning_rate': 4.8211731044349076e-05, 'epoch': 0.11}


e:\anaconda3\envs\peft\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
                                                    
  4%|▍         | 110/2796 [04:30<1:55:09,  2.57s/it]

{'loss': 3.6508, 'grad_norm': 11.451169967651367, 'learning_rate': 4.803290414878398e-05, 'epoch': 0.12}


  4%|▍         | 111/2796 [04:33<1:56:51,  2.61s/it]

KeyboardInterrupt: 